In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import os
from PIL import Image
import pandas as pd
from tqdm import tqdm
import torch.backends.cudnn as cudnn
import matplotlib.pyplot as plt
import numpy as np

cudnn.benchmark = True

In [2]:
class TinyImageNetDataset(Dataset):
    def __init__(self, root_dir, transform=None, train=True):
        self.root_dir = root_dir

        self.train = train
        self.transform = transform

        if self.train:
            self.data = []
            self.labels = []
            classes = sorted(os.listdir(os.path.join(root_dir, 'train')))

            # Поиск картинок и присвоение labels
            for label, cls in enumerate(classes):
                cls_dir = os.path.join(root_dir, 'train', cls, 'images')
                for img_name in os.listdir(cls_dir):
                    self.data.append(os.path.join(cls_dir, img_name)) # store the path only
                    self.labels.append(label)
        else:
            self.data = []
            self.labels = []
            val_dir = os.path.join(root_dir, 'val', 'images')

            # Чтение csv файлов с данными
            val_annotations = pd.read_csv(os.path.join(root_dir, 'val', 'val_annotations.txt'),
                                          sep='\t', header=None,
                                          names=['file_name', 'class', 'x1', 'y1', 'x2', 'y2'])
            class_to_idx = {cls: idx for idx, cls in enumerate(sorted(os.listdir(os.path.join(root_dir, 'train'))))}
            for _, row in val_annotations.iterrows():
                self.data.append(os.path.join(val_dir, row['file_name']))
                self.labels.append(class_to_idx[row['class']])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data[idx]
        # Загрузка изображения
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        # Применение трансформаций
        if self.transform:
            image = self.transform(image)

        return image, label

In [3]:
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, k_size, stride, expand_ratio, se_ratio = 0.25):
        super().__init__()
        self.stride = stride
        self.expand_ratio = expand_ratio
        self.se_ratio = se_ratio

        hidden_dim = in_channels * expand_ratio
        #expansion
        if expand_ratio > 1:
            self.expand = nn.Sequential(
                nn.Conv2d(in_channels=in_channels, out_channels=hidden_dim, kernel_size = 1, bias = False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace = True)
        )
        else:
            self.expand = None

        #depthwise convolution
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(in_channels=hidden_dim, out_channels=hidden_dim, kernel_size=k_size, stride=stride, padding = (k_size - 1) // 2, groups=hidden_dim, bias = False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace = True)
        )

        #se
        se_hidden_dim = max(1, int(self.se_ratio * in_channels))
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Conv2d(in_channels=hidden_dim, out_channels=se_hidden_dim, kernel_size=1, bias = False),
            nn.SiLU(inplace = True),
            nn.Conv2d(in_channels= se_hidden_dim, out_channels=hidden_dim, kernel_size=1, bias = False),
            nn.Sigmoid()
        )

        #output
        self.output = nn.Sequential(
            nn.Conv2d(in_channels=hidden_dim, out_channels=out_channels, kernel_size=1, bias = False),
            nn.BatchNorm2d(out_channels)
        )
        self.use_res_con = (in_channels == out_channels and stride == 1)

    def forward(self, X):
        out = X
        if self.expand:
            out = self.expand(out)
        out = self.depthwise_conv(out)
        out = out * self.se(out) 
        out = self.output(out)
        if self.use_res_con:
            out = out + X
        return out
        
class EfficientNetB0(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stage1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding = 1, bias = False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace = True)
        )
        self.stage2 = self.__make_layer__(in_channels = 32, out_channels = 16, kernel_size = 3, stride = 1, num_layers = 1, expand_ratio = 1)
        self.stage3 = self.__make_layer__(in_channels = 16, out_channels = 24, kernel_size = 3, stride = 2, num_layers = 2, expand_ratio = 6)
        self.stage4 = self.__make_layer__(in_channels = 24, out_channels = 40, kernel_size = 5, stride = 2, num_layers = 2, expand_ratio = 6)
        self.stage5 = self.__make_layer__(in_channels = 40, out_channels = 80, kernel_size = 3, stride = 2, num_layers = 3, expand_ratio = 6)
        self.stage6 = self.__make_layer__(in_channels = 80, out_channels = 112, kernel_size = 5, stride = 1, num_layers = 3, expand_ratio = 6)
        self.stage7 = self.__make_layer__(in_channels = 112, out_channels = 192, kernel_size = 5, stride = 2, num_layers = 4, expand_ratio = 6)
        self.stage8 = self.__make_layer__(in_channels = 192, out_channels = 320, kernel_size = 3, stride = 1, num_layers = 1, expand_ratio = 6)
        self.stage9 = nn.Sequential(
            nn.Conv2d(320, 320, kernel_size=1),
            nn.Dropout2d(0.5),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(320, 1280),
            nn.ReLU(),
            nn.Linear(1280, num_classes)
        )
        
    def __make_layer__(self, in_channels, out_channels, kernel_size, stride, num_layers, expand_ratio):
        layers = []
        layers.append(MBConv(in_channels=in_channels, out_channels=out_channels, k_size=kernel_size, stride=stride, expand_ratio=expand_ratio))
        for layer in range(1, num_layers):
            layers.append(MBConv(in_channels=out_channels, out_channels=out_channels, k_size=kernel_size, stride=1, expand_ratio=expand_ratio))
        return nn.Sequential(*layers)
        
    def forward(self, X):
        out = self.stage1(X)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        out = self.stage5(out)
        out = self.stage6(out)
        out = self.stage7(out)
        out = self.stage8(out)
        out = self.stage9(out)
        return out
        

In [10]:
val_transform = transforms.Compose(
    [transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
)

train_transform = transforms.Compose(
    [transforms.RandomGrayscale(),
     transforms.RandomHorizontalFlip(),
     transforms.ToTensor(),
     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
)

train_data = TinyImageNetDataset(root_dir = r'C:\DATA\tiny-imagenet-200', transform=train_transform, train=True)
val_data = TinyImageNetDataset(root_dir = r'C:\DATA\tiny-imagenet-200', transform=val_transform, train=False)

BATCH_SIZE = 16
train_loader = DataLoader(train_data, batch_size = BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size= BATCH_SIZE, shuffle=False, num_workers=0)

In [11]:
epochs = 25
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = EfficientNetB0(num_classes=1000).to(device) # torchvision.models.efficientnet_b0().to(device) 
learning_rate = 0.01
optimizer = optim.SGD(params=model.parameters(), lr=learning_rate, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, eta_min = 1e-5, T_max=6250)


In [12]:
for epoch in range(epochs):
    train_loop = tqdm(train_loader)
    train_loss = []
    model.train()
    for X, y in train_loop:
        X = X.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss.append(loss)
        loss.backward() 
        optimizer.step()
        scheduler.step()
    print(f'Epoch {epoch + 1}/{epochs}, Loss {(sum(train_loss) / len(train_loss))}, LR {scheduler.get_last_lr()[0]}')
    
    with torch.no_grad():
        model.eval()
        correct = 0
        total = 0
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)
            y_pred = model(X)
            total += y.size(0)
            _, pred = torch.max(y_pred, 1)
            correct += (pred == y).sum().item()
            acc = 100. * correct / total
    print(f'Accuracy {acc:.4f}')

100%|██████████████████████████████████████████████████████████████████████████████| 6250/6250 [04:07<00:00, 25.29it/s]


Epoch 1/25, Loss 5.325993537902832, LR 1e-05
Accuracy 1.7800


100%|██████████████████████████████████████████████████████████████████████████████| 6250/6250 [04:17<00:00, 24.26it/s]


Epoch 2/25, Loss 5.010924339294434, LR 0.009999999999996928
Accuracy 4.3100


100%|██████████████████████████████████████████████████████████████████████████████| 6250/6250 [04:31<00:00, 23.01it/s]


...

Epoch 23/25, Loss 2.762507200241089, LR 1e-05
Accuracy 34.9700


100%|██████████████████████████████████████████████████████████████████████████████| 6250/6250 [04:21<00:00, 23.91it/s]


Epoch 24/25, Loss 2.638127326965332, LR 0.009999999999997032
Accuracy 30.3900


100%|██████████████████████████████████████████████████████████████████████████████| 6250/6250 [04:21<00:00, 23.88it/s]


Epoch 25/25, Loss 2.6832118034362793, LR 1e-05
Accuracy 35.6800
